# 04 · Beyond the basics: Dash, HoloViews, Panel and Datashader

A **survey**: what to reach for when Streamlit or plain Plotly/Altair stop being enough. Two kinds of limits:

1. **Control.** You need precise layout, many interacting components, or a production app with predictable performance → **Dash** (or Panel).
2. **Scale.** You have millions of points and the browser chokes → **Datashader**, usually through **HoloViews / hvPlot** and **Panel** (the HoloViz ecosystem).

## Part A . Dash: explicit callbacks

Streamlit reruns the whole script on every interaction. Dash instead asks you to declare **callbacks**: functions that run when specific inputs change and update specific outputs. More code, but only what's needed reruns, and complex apps stay predictable.

| | Streamlit | Dash |
| --- | --- | --- |
| Execution model | Rerun whole script | Callbacks: inputs → outputs |
| Layout | Linear page, columns, tabs | Full HTML/CSS control (`html.Div`, CSS grid) |
| Charts | Any library | Plotly (`dcc.Graph`) first-class |
| Chart events | `on_select="rerun"` | `hoverData`, `clickData`, `selectedData`, `relayoutData` |
| Learning curve | Minutes | An afternoon |
| Good for | Internal tools, prototypes | Production dashboards, complex interaction |

This app has a dropdown and slider that update the scatter, and **hovering** a country on the scatter updates the line chart: cross-filtering driven by chart events. `app.run(jupyter_mode="inline")` shows a Dash app right in the notebook.

In [ ]:
# Execute these lines if running on the JupyterHub
# import os
# prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "/")   # e.g. /user/<you>/

In [ ]:
"""Gapminder in Dash: explicit callbacks.  Run with:  python dash_gapminder.py"""
import plotly.express as px
from dash import Dash, Input, Output, dcc, html

gap = px.data.gapminder()

# Local computer:
app = Dash(__name__)
# In JupyterHub:
# app = Dash(__name__,
#            routes_pathname_prefix="/",
#            requests_pathname_prefix=f"{prefix}proxy/8050/")

app.layout = html.Div([
    html.H3("Gapminder in Dash"),
    html.Div([
        dcc.Dropdown(sorted(gap.continent.unique()), "Asia", id="continent",
                     clearable=False, style={"width": "200px"}),
        dcc.Slider(1952, 2007, step=5, value=2007, id="year",
                   marks={y: str(y) for y in range(1952, 2008, 10)}),
    ], style={"display": "grid", "gridTemplateColumns": "220px 1fr", "gap": "16px"}),
    html.Div([
        dcc.Graph(id="scatter", style={"flex": "3"}),
        dcc.Graph(id="history", style={"flex": "2"}),
    ], style={"display": "flex"}),
], style={"fontFamily": "sans-serif", "padding": "8px"})


# Callback 1: two widgets -> scatter figure
@app.callback(Output("scatter", "figure"),
              Input("continent", "value"), Input("year", "value"))
def update_scatter(continent, year):
    d = gap[(gap.continent == continent) & (gap.year == year)]
    fig = px.scatter(d, x="gdpPercap", y="lifeExp", size="pop", hover_name="country",
                     custom_data=["country"], log_x=True, size_max=45,
                     range_y=[25, 90], title=f"{continent}, {year} (hover a country)")
    return fig


# Callback 2: hovering on the scatter -> line chart of that country
@app.callback(Output("history", "figure"),
              Input("scatter", "hoverData"), Input("continent", "value"))
def update_history(hover, continent):
    if hover:
        country = hover["points"][0]["customdata"][0]
    else:
        country = gap[gap.continent == continent].country.iloc[0]
    d = gap[gap.country == country]
    return px.line(d, x="year", y="lifeExp", markers=True, range_y=[25, 90],
                   title=country)


# Local computer:
app.run(jupyter_mode="inline", port=8050, jupyter_height=560)
# In JupyterHub:
# app.run(jupyter_mode="inline", port=8050, jupyter_height=560,
#         jupyter_server_url="https://ucla-oarc-workshop.nrp-nautilus.io")   # origin only: no /user/... and no trailing slash

Notice the callback signatures: `Output("scatter", "figure")` says *which property of which component* this function produces. Dash builds a dependency graph from these declarations.

The same app as a standalone script is in `apps/dash_gapminder.py` (`python apps/dash_gapminder.py`).

## Part B . When the data is too big to draw point by point

Plotly and Altair send every point to the browser as JSON and draw each one. At a few hundred thousand points the page slows; at millions it crashes. Even when it works, **overplotting** hides the structure: a million overlapping dots are just a blob.

**Datashader** flips the approach: it *aggregates* points into a pixel grid in Python (fast, using Numba), then sends an **image**. Five million points cost the same to display as five hundred.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
centers = {"A": (-2, -1, 0.6), "B": (1.5, 2, 0.4), "C": (2, -2, 0.9), "D": (0, 0, 2.5)}
n_each = 1_250_000
big = pd.concat([
    pd.DataFrame({"x": rng.normal(cx, s, n_each), "y": rng.normal(cy, s, n_each), "group": g})
    for g, (cx, cy, s) in centers.items()
], ignore_index=True)
big["group"] = big["group"].astype("category")
print(f"{len(big):,} rows, {big.memory_usage(deep=True).sum() / 1e6:.0f} MB")

### The overplotting problem

Here is a 2% sample in matplotlib. Group B (a tight cluster of 1.25 million points) looks no denser than the diffuse background.

In [ ]:
import matplotlib.pyplot as plt

s = big.sample(100_000, random_state=0)
plt.figure(figsize=(5, 4))
plt.scatter(s.x, s.y, s=1, alpha=0.3)
plt.title("100k-point sample, overplotted")
plt.show()

### Datashader's pipeline: canvas → aggregate → shade

In [ ]:
import datashader as ds
import datashader.transfer_functions as tf

canvas = ds.Canvas(plot_width=500, plot_height=400)
agg = canvas.points(big, "x", "y", agg=ds.count())   # a 400x500 array of counts
tf.shade(agg, how="eq_hist", cmap=["#fff7bc", "#fe9929", "#662506"])

`how="eq_hist"` (histogram equalization) reveals structure across a huge range of densities. Compare `how="linear"` (only the densest cluster is visible) and `how="log"`.

Aggregate by **category** to color by group:

In [ ]:
agg_cat = canvas.points(big, "x", "y", agg=ds.count_cat("group"))
colors = {"A": "#1b9e77", "B": "#d95f02", "C": "#7570b3", "D": "#e7298a"}
tf.shade(agg_cat, color_key=colors, how="eq_hist")

### Making it interactive: hvPlot + HoloViews

A static image loses zoom. **hvPlot** (a `.hvplot` accessor on DataFrames, built on HoloViews and Bokeh) re-runs Datashader **every time you zoom or pan**, so detail appears as you zoom in. Only an image ever reaches the browser.

*This needs a live kernel: in an exported HTML file the image won't re-render on zoom.*

In [ ]:
import holoviews as hv
import hvplot.pandas  # noqa: F401  (adds .hvplot)
import panel as pn

hv.extension("bokeh")
pn.extension(inline=True)   # only needed if you still get the "Panel could not load resources" error

big.hvplot.points("x", "y", rasterize=True, cnorm="eq_hist", cmap="viridis",
                  width=550, height=420, colorbar=True,
                  title="5,000,000 points: zoom in!")

hvPlot speaks the same `.plot()` vocabulary as pandas, so small data is one keyword away from a familiar interactive Bokeh chart. HoloViews objects compose with `+` (side by side) and `*` (overlay):

In [ ]:
import plotly.express as px

gap = px.data.gapminder()
g07 = gap.query("year == 2007")

scatter = g07.hvplot.scatter("gdpPercap", "lifeExp", by="continent", logx=True,
                             hover_cols=["country"], height=320, width=420)
hist = g07.hvplot.hist("lifeExp", bins=20, height=320, width=300)
scatter + hist

`hv.link_selections` adds linked brushing across HoloViews plots, including rasterized ones over millions of points, with the filtering done in Python. Box-select on either plot:

In [ ]:
ls = hv.link_selections.instance()
ls(big.hvplot.points("x", "y", rasterize=True, cnorm="eq_hist", width=380, height=320)
   + big.hvplot.hist("x", bins=60, width=320, height=320))

### Panel: dashboards from any of these objects

**Panel** turns widgets plus *any* plot (hvPlot, Plotly, Altair, matplotlib) into an app. It works inside the notebook and as a server: `panel serve app.py`. `pn.bind` connects widgets to a function, much like a Streamlit widget feeding a function, but only the bound function reruns.

In [ ]:
import panel as pn

pn.extension()

groups = pn.widgets.MultiChoice(name="Groups", options=list("ABCD"), value=list("ABCD"))
norm = pn.widgets.RadioButtonGroup(name="Color scale", options=["eq_hist", "log", "linear"])

def plot(groups, norm):
    d = big[big.group.isin(groups)]
    return d.hvplot.points("x", "y", rasterize=True, cnorm=norm, cmap="viridis",
                           width=550, height=420, title=f"{len(d):,} points")

pn.Row(pn.Column(groups, norm, width=220), pn.bind(plot, groups, norm))

The same app as a deployable script is `apps/panel_bigdata_app.py`:

```bash
panel serve apps/panel_bigdata_app.py --show
```
**OR ON JUPYTERHUB**
```bash
panel serve apps/panel_bigdata_app.py --allow-websocket-origin=ucla-oarc-workshop.nrp-nautilus.io
```

## Which tool when?

| Situation | Reach for |
| --- | --- |
| Explore in a notebook, share a chart | Plotly Express or Altair |
| Linked views that must work in a plain HTML file | Altair (or HoloViews, for small data) |
| Quick internal dashboard, mostly linear layout | Streamlit |
| Production dashboard, custom layout, many interacting parts | Dash (or Panel) |
| Over ~1M points, maps of dense data, big time series | Datashader via hvPlot/HoloViews |
| Dashboard over big data, or mixing many plotting libraries | Panel + hvPlot |

Middle ground before Datashader: Plotly's WebGL scatter handles around 1M points, and `plotly-resampler` downsamples long time series on zoom.

---

### ✏️ Exercise: Optional: extend the Dash app

Copy the Dash cell above and add a third callback: **clicking** a country (`Input("scatter", "clickData")`) writes its name and 2007 population into an `html.Div(id="info")` placed under the dropdown. Use a different port (e.g. 8051) so it doesn't clash.

In [ ]:
# Your code here

### ✏️ Exercise: Optional: Datashader on real-looking data

Make a time series that's too long to plot: `t = np.arange(10_000_000)` and `y = np.cumsum(rng.normal(size=t.size))` (a random walk). Plot it with `pd.DataFrame({"t": t, "y": y}).hvplot.line("t", "y", rasterize=True)` and zoom into a small window.

In [ ]:
# Your code here